# U7-01 · Bandidos, procesos de Markov e iteración de valores

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 7, apartados 7.2 a 7.4*

Ejecuta cada celda con **Mayús + Intro**. Solo se usa NumPy.

## 1. Tres configuraciones de una máquina (ejemplo 1)

Cada configuración da un lote bueno con probabilidad 0,55, 0,60 o 0,70. El agente no lo sabe.

In [ ]:
import numpy as np

def jugar(estrategia, p=(0.55, 0.60, 0.70), pasos=1000, rng=None):
    Q, N = np.zeros(3), np.zeros(3)      # estimaciones y conteos
    total, mejor = 0, 0
    for t in range(pasos):
        if t < 3:
            a = t                                  # probar cada una
        elif estrategia == "eps" and rng.random() < 0.1:
            a = int(rng.integers(3))               # explorar
        elif estrategia == "ucb":
            a = int(np.argmax(Q + np.sqrt(np.log(t + 1) / N)))
        else:
            a = int(np.argmax(Q))                  # explotar
        r = float(rng.random() < p[a])
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]                  # media incremental
        total, mejor = total + r, mejor + (a == 2)
    return total, mejor

Repetimos 200 veces cada estrategia (en el libro, 2000; tarda más). Devuelve los lotes buenos de 1000 y las veces que eligió la mejor configuración.

In [ ]:
rng = np.random.default_rng
for e in ["voraz", "eps", "ucb"]:
    res = [jugar(e, rng=rng(i)) for i in range(200)]
    print(f"{e:>5}: {np.mean(res, axis=0)}")

**Prueba.** Cambia `0.1` por `0.01` y por `0.3` en la línea de la estrategia épsilon-voraz. ¿Qué pasa con los lotes buenos? Explorar demasiado también cuesta.

## 2. El mantenimiento de una máquina (ejemplos 2 y 3)

Iteración de valores: se aplica la ecuación de Bellman hasta que los valores dejan de cambiar.

In [ ]:
import numpy as np
# estados: 0 nueva, 1 desgaste leve, 2 desgaste fuerte, 3 averiada
# acciones: 0 producir, 1 revisar (en averiada, ambas = reparar)
P = np.zeros((2, 4, 4))
P[0] = [[.7, .3, 0, 0], [0, .6, .4, 0], [0, 0, .5, .5], [1, 0, 0, 0]]
P[1] = [[1, 0, 0, 0]] * 4
R = np.array([[10, 8, 5, -40],       # producir
              [-6, -6, -6, -40]])    # revisar
gamma = 0.9
V = np.zeros(4)
for k in range(1000):
    Q = R + gamma * P @ V            # Q[a, s]: acción a en estado s
    V_nuevo = Q.max(axis=0)
    if np.abs(V_nuevo - V).max() < 1e-10:
        break
    V = V_nuevo
print(k + 1, V_nuevo.round(2), Q.argmax(axis=0))

Comprobamos la política simulando muchos periodos y comparándola con otras dos.

In [ ]:
def simular(pol, periodos=200_000, semilla=0):
    rng = np.random.default_rng(semilla)
    s, total = 0, 0.0
    for t in range(periodos):
        a = pol[s]
        total += R[a, s]
        s = rng.choice(4, p=P[a, s])
    return total / periodos

for nombre, pol in [("no revisar nunca", [0, 0, 0, 0]),
                    ("revisar con desgaste leve", [0, 1, 1, 0]),
                    ("óptima", list(Q.argmax(axis=0)))]:
    print(f"{nombre:>26}: {simular(pol):.2f} por periodo")

**Ejercicio.** Sube el coste de la revisión de 6 a 15 (las dos `-6` de la fila de `revisar`) y vuelve a ejecutar. ¿Sigue compensando revisar? ¿Y si la reparación cuesta 80?

> Sutton, R. S. y Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2.ª ed.). MIT Press. ISBN 978-0-262-03924-6
>
> Auer, P., Cesa-Bianchi, N. y Fischer, P. (2002). Finite-time analysis of the multiarmed bandit problem. *Machine Learning*, 47, 235–256. https://doi.org/10.1023/A:1013689704352